# CFD Surrogate - Colab runner

This notebook is only an orchestrator. It does not duplicate the training/export code from the repo; it installs the Colab runtime dependencies, configures OpenFOAM CLI, then executes the checked-in Python scripts directly.

Default flow:

1. Mount Drive and locate this repo plus `input/` OpenFOAM cases.
2. Install Python dependencies needed by PhysicsNeMo, PyVista, PyG, and TorchScript export.
3. Install OpenFOAM 13 CLI and verify `foamToVTK`.
4. Run `01_export_openfoam.py` through a shell that has sourced OpenFOAM.
5. Run `02_train_mgn.py` or `02_train_fno.py` directly.
6. Run `evaluate.py` for MGN outputs, export `cfd_surrogate.ts`, and package the output zip.

Set the variables in the next cell before `Runtime -> Run all`.


In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('Not running in Colab; continuing without Drive mount.')

# Optional: set this if the notebook is opened without the repo files beside it.
# Example: REPO_URL = 'https://github.com/<owner>/<repo>.git'
REPO_URL = ''
REPO_DIR = Path('/content/work')
DRIVE_REPO_CANDIDATES = [
    Path('/content/drive/MyDrive/work'),
    Path('/content/drive/MyDrive/cfd-surrogate'),
    Path('/content/drive/MyDrive/CFD'),
]

# Data/config.
INPUT_DIR = None  # None = auto-discover a folder containing OpenFOAM case.foam files.
ARCH = 'mgn'      # 'mgn' or 'fno'
EPOCHS = 3000
SUBSAMPLE = 300_000
INFER_FRAMES = 21
PI_WEIGHT = 0.0
GRID_RES = 64     # FNO only

# OpenFOAM export. Keep VDB disabled unless pyopenvdb is installed separately.
INSTALL_OPENFOAM = True
OPENFOAM_VERSION = '13'
RUN_OPENFOAM_EXPORT = True
EXPORT_VDB = False

OUT_MGN = Path('/content/out_outputs')
OUT_FNO = Path('/content/out_fno')
OUT_GT = Path('/content/out_gt')
PACKAGE_PATH = Path('/content/colab_outputs.zip')


In [ ]:
def q(value) -> str:
    return shlex.quote(str(value))


def run(cmd: str, *, openfoam: bool = False) -> None:
    if openfoam:
        bashrc = f'/opt/openfoam{OPENFOAM_VERSION}/etc/bashrc'
        cmd = f'test -f {q(bashrc)} && source {q(bashrc)} && {cmd}'
    print(f'\n$ {cmd}')
    subprocess.run(['bash', '-lc', cmd], check=True)


def has_repo_files(path: Path) -> bool:
    return (path / '02_train_mgn.py').exists() and (path / '03_export_model.py').exists()


cwd = Path.cwd()
if has_repo_files(cwd):
    REPO_DIR = cwd
elif has_repo_files(REPO_DIR):
    pass
elif REPO_URL:
    if not REPO_DIR.exists():
        run(f'git clone {q(REPO_URL)} {q(REPO_DIR)}')
else:
    for candidate in DRIVE_REPO_CANDIDATES:
        if has_repo_files(candidate):
            REPO_DIR = candidate
            break

if not has_repo_files(REPO_DIR):
    raise FileNotFoundError(
        'Repo scripts were not found. Upload/copy the repo to Drive, set REPO_URL, '
        'or run this notebook from a directory containing 02_train_mgn.py.'
    )

os.chdir(REPO_DIR)
print('REPO_DIR =', Path.cwd())


## Install Python dependencies

This keeps Colab's installed PyTorch/CUDA pair and installs PyG wheels that match it. That is usually more reliable on Colab than forcing the local `requirements.txt` torch pins.


In [ ]:
run(f'{q(sys.executable)} -m pip install -q --upgrade pip')
run(f'{q(sys.executable)} -m pip install -q nvidia-physicsnemo pyvista scipy')

import torch
TV = torch.__version__.split('+')[0]
CU = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'wheel tag', f'{TV}+{CU}')

run(f'{q(sys.executable)} -m pip install -q torch_geometric')
run(f'{q(sys.executable)} -m pip install -q torch_scatter -f https://data.pyg.org/whl/torch-{TV}+{CU}.html')
run(f'{q(sys.executable)} -m py_compile cfd_cases.py 01_export_openfoam.py 02_train_mgn.py 02_train_fno.py 03_export_model.py evaluate.py')


## Install and verify OpenFOAM CLI

`01_export_openfoam.py` calls `foamToVTK`, so OpenFOAM must be installed and sourced before that script runs. This cell installs OpenFOAM 13 on Colab's Ubuntu runtime and verifies the CLI.


In [ ]:
if INSTALL_OPENFOAM:
    run('sudo apt-get update -qq')
    run('sudo apt-get install -y -qq wget software-properties-common')
    run('sudo apt-add-repository -y universe >/dev/null')
    run('sudo sh -c "wget -qO - https://dl.openfoam.org/gpg.key > /etc/apt/trusted.gpg.d/openfoam.asc"')
    run('sudo add-apt-repository -y http://dl.openfoam.org/ubuntu >/dev/null')
    run('sudo apt-get update -qq')
    run(f'sudo apt-get install -y -qq openfoam{OPENFOAM_VERSION}')

run('command -v foamToVTK && foamToVTK -help | head -20', openfoam=True)


## Locate OpenFOAM input cases


In [ ]:
def contains_cases(path: Path) -> bool:
    return path.exists() and any(p.name == 'case.foam' for p in path.rglob('case.foam'))


if INPUT_DIR is None:
    candidates = [
        Path.cwd() / 'input',
        Path('/content/input'),
        Path('/content/drive/MyDrive/input'),
        Path('/content/drive/MyDrive/work/input'),
    ]
    for candidate in candidates:
        if contains_cases(candidate):
            INPUT_DIR = candidate
            break

if INPUT_DIR is None or not contains_cases(Path(INPUT_DIR)):
    raise FileNotFoundError('No OpenFOAM case.foam files found. Set INPUT_DIR to your input folder.')

INPUT_DIR = Path(INPUT_DIR).resolve()
print('INPUT_DIR =', INPUT_DIR)
run(f'{q(sys.executable)} -c "from cfd_cases import discover_cases; print([(c.omega, round(c.rpm), c.path.name) for c in discover_cases({str(INPUT_DIR)!r})])"')


## Step 1 - Export OpenFOAM ground truth through `foamToVTK`


In [ ]:
if RUN_OPENFOAM_EXPORT:
    vdb_flag = '' if EXPORT_VDB else '--no-vdb'
    run(
        f'{q(sys.executable)} 01_export_openfoam.py --input {q(INPUT_DIR)} --out {q(OUT_GT)} {vdb_flag}',
        openfoam=True,
    )
else:
    print('Skipped OpenFOAM export because RUN_OPENFOAM_EXPORT=False')


## Step 2 - Train surrogate by executing the repo script


In [ ]:
if ARCH == 'mgn':
    OUT_DIR = OUT_MGN
    train_cmd = (
        f'{q(sys.executable)} 02_train_mgn.py '
        f'--input {q(INPUT_DIR)} --out {q(OUT_DIR)} '
        f'--epochs {int(EPOCHS)} --subsample {int(SUBSAMPLE)} '
        f'--infer-frames {int(INFER_FRAMES)} --pi-weight {float(PI_WEIGHT)}'
    )
elif ARCH == 'fno':
    OUT_DIR = OUT_FNO
    train_cmd = (
        f'{q(sys.executable)} 02_train_fno.py '
        f'--input {q(INPUT_DIR)} --out {q(OUT_DIR)} '
        f'--epochs {int(EPOCHS)} --grid-res {int(GRID_RES)} '
        f'--infer-frames {int(INFER_FRAMES)} --pi-weight {float(PI_WEIGHT)}'
    )
else:
    raise ValueError("ARCH must be 'mgn' or 'fno'")

run(train_cmd)


## Step 3 - Evaluate and export TorchScript


In [ ]:
if ARCH == 'mgn':
    run(
        f'{q(sys.executable)} evaluate.py '
        f'--input {q(INPUT_DIR)} '
        f'--predictions {q(OUT_DIR / "predictions.npz")} '
        f'--sub-idx {q(OUT_DIR / "sub_idx.npy")} '
        f'--output {q(OUT_DIR / "evaluation_metrics.json")}'
    )
else:
    print('Skipping evaluate.py for FNO: current evaluator expects flat sampled-cell MGN arrays.')

run(
    f'{q(sys.executable)} 03_export_model.py '
    f'--arch {q(ARCH)} --out-dir {q(OUT_DIR)} '
    f'--out {q(OUT_DIR / "cfd_surrogate.ts")} --device cpu'
)


## Step 4 - Package outputs


In [ ]:
if PACKAGE_PATH.exists():
    PACKAGE_PATH.unlink()
shutil.make_archive(str(PACKAGE_PATH.with_suffix('')), 'zip', str(OUT_DIR))
print(f'Packaged {PACKAGE_PATH} ({PACKAGE_PATH.stat().st_size / 1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(str(PACKAGE_PATH))
except ModuleNotFoundError:
    print('Not in Colab; download step skipped.')
